In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

In [11]:
class StateAgent(TypedDict):
    name: str
    marks:int
    attendance:int
    marks_eligible: bool
    attendance_eligible: bool
    result: str

In [15]:
def check_marks(state:StateAgent):
  if state["marks"] >=80:
    state["marks_eligible"]=True
  else:
    state["marks_eligible"]=False
  return state
def attendance_eligible(state:StateAgent):
  "chceking the attendance eligibility"
  if state["attendance"] >=75:
    state["attendance_eligible"]=True
  else:
    state["attendance_eligible"]=False
  return state
def router(state:StateAgent):
  if (
      state["marks_eligible"] == True
        and state["attendance_eligible"] == True
  ):
    return "eligible"
  else:
    return "not_eligible"
def give_scholarship(state:StateAgent):
    state["result"] = (
        f"{state['name']} is eligible for the scholarship."
    )

    return state
def reject_scholarship(state: StateAgent):
    state["result"] = (
        f"{state['name']} is not eligible for the scholarship."
    )

    return state



In [32]:
graph = StateGraph(StateAgent)



graph.add_node("check_marks", check_marks)
graph.add_node("attendance_eligible", attendance_eligible)

graph.add_node("give_scholarship", give_scholarship)
graph.add_node("reject_scholarship", reject_scholarship)



graph.add_edge(START, "check_marks")
graph.add_edge("check_marks", "attendance_eligible")



graph.add_conditional_edges(
    "attendance_eligible",
    router,
    {
        "eligible": "give_scholarship",
        "not_eligible": "reject_scholarship",
    }
)


graph.add_edge("give_scholarship", END)
graph.add_edge("reject_scholarship", END)


In [34]:
app = graph.compile()
input_data = {
    "name": "Anushka",
    "marks": 50,
    "attendance": 90,
    "marks_eligible": False,
    "attendance_eligible": False,
    "result": "",
}

result = app.invoke(input_data)

print(result["result"])

Anushka is not eligible for the scholarship.


InvalidUpdateError: Expected dict, got eligible
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE